In [15]:
# from 
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
# from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [16]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
model = ChatGroq(
    model="openai/gpt-oss-20b", api_key=api_key,
    temperature=0)

In [17]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [18]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content

    return {'joke': response}

In [19]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = model.invoke(prompt).content

    return {'explanation': response}

In [20]:
# Create a StateGraph using JokeState to define the workflow's state
graph = StateGraph(JokeState)

# Add a node that generates a joke
# 'generate_joke' is the node name, and generate_joke is the function
graph.add_node('generate_joke', generate_joke)

# Add a node that explains the generated joke
graph.add_node('generate_explanation', generate_explanation)

# Define the starting point of the workflow
# The workflow will execute the generate_joke node first
graph.add_edge(START, 'generate_joke')

# Connect the joke-generation node to the explanation node
# After generating the joke, the workflow explains it
graph.add_edge('generate_joke', 'generate_explanation')

# Define the ending point of the workflow
# The workflow finishes after explaining the joke
graph.add_edge('generate_explanation', END)

# Create an in-memory checkpointer to save workflow checkpoints
# Checkpoints are lost when the Python process stops
checkpointer = InMemorySaver()

# Compile the graph into an executable workflow
# Attach the checkpointer to support state persistence during execution
workflow = graph.compile(checkpointer=checkpointer)

In [21]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza start a band?\n\nBecause it already had the perfect *crust* for a *cheesy* rhythm! 🍕🎸',
 'explanation': ''}

In [22]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza start a band?\n\nBecause it already had the perfect *crust* for a *cheesy* rhythm! 🍕🎸', 'explanation': ''}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-65da-6ef2-8002-9024971ce05e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-27T04:48:00.681137+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-4897-690d-8001-aeb95c0ac5e1'}}, tasks=(), interrupts=())

In [23]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza start a band?\n\nBecause it already had the perfect *crust* for a *cheesy* rhythm! 🍕🎸', 'explanation': ''}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-65da-6ef2-8002-9024971ce05e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-27T04:48:00.681137+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-4897-690d-8001-aeb95c0ac5e1'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza start a band?\n\nBecause it already had the perfect *crust* for a *cheesy* rhythm! 🍕🎸'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-4897-690d-8001-aeb95c0ac5e1'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-10-27T04:47:57.612673+00:00', parent_config={'conf

In [24]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti start a band?\n\nBecause it wanted to be a *fettuccine* rockstar—every time it hit a note, it *pasta* the beat!',
 'explanation': '**Why the joke works – a quick “pasta‑pun” breakdown**\n\n| Part of the joke | What it’s playing on | Why it’s funny |\n|------------------|----------------------|----------------|\n| **“Why did the spaghetti start a band?”** | Sets up a classic “why did X do Y?” joke format. | The listener expects a logical answer, but the punchline will be a pun. |\n| **“Because it wanted to be a *fettuccine* rockstar”** | *Fettuccine* is a type of pasta; “rockstar” is a musician. | The humor comes from treating a pasta shape as a music‑industry title. The word *fettuccine* sounds like “fettuccine rockstar” – a mash‑up of food and fame. |\n| **“—every time it hit a note, it *pasta* the beat!”** | *Pasta* is a noun (the food) but sounds like “pass” or “pasta” as a verb. | The phrase “hit a note” is a musical idiom. Replaci

In [25]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza start a band?\n\nBecause it already had the perfect *crust* for a *cheesy* rhythm! 🍕🎸', 'explanation': ''}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-65da-6ef2-8002-9024971ce05e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-27T04:48:00.681137+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-4897-690d-8001-aeb95c0ac5e1'}}, tasks=(), interrupts=())

In [26]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza start a band?\n\nBecause it already had the perfect *crust* for a *cheesy* rhythm! 🍕🎸', 'explanation': ''}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-65da-6ef2-8002-9024971ce05e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-27T04:48:00.681137+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-4897-690d-8001-aeb95c0ac5e1'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza start a band?\n\nBecause it already had the perfect *crust* for a *cheesy* rhythm! 🍕🎸'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-4897-690d-8001-aeb95c0ac5e1'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-10-27T04:47:57.612673+00:00', parent_config={'conf

### Time Travel

In [27]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [28]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

EmptyInputError: Received no input for __start__

In [29]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza start a band?\n\nBecause it already had the perfect *crust* for a *cheesy* rhythm! 🍕🎸', 'explanation': ''}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-65da-6ef2-8002-9024971ce05e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-27T04:48:00.681137+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-4897-690d-8001-aeb95c0ac5e1'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza start a band?\n\nBecause it already had the perfect *crust* for a *cheesy* rhythm! 🍕🎸'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-4897-690d-8001-aeb95c0ac5e1'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-10-27T04:47:57.612673+00:00', parent_config={'conf

#### Updating State

In [30]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1d1c27-2002-6f4d-8000-7b11f0bcde90'}}

In [31]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c27-2002-6f4d-8000-7b11f0bcde90'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-10-27T04:54:09.167137+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='6360c9be-bbd9-aa65-c3c6-0d92575c72fb', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza start a band?\n\nBecause it already had the perfect *crust* for a *cheesy* rhythm! 🍕🎸', 'explanation': ''}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1c19-65da-6ef2-8002-9024971ce05e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-

In [32]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc72-ca16-6359-8001-7eea05e07dd2"}})

EmptyInputError: Received no input for __start__

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa bring a ladder to the party? \nBecause it wanted to be the best snack in the room and rise to the occasion!', 'explanation': 'This joke plays on the double meaning of the word "rise." In one sense, "rise" means to physically move upwards, which is why the samosa brought a ladder to the party. However, in another sense, "rise" can also mean to perform well or excel, as in rising to the occasion. So, the samosa brought a ladder to symbolize its desire to physically rise above the other snacks at the party and also to metaphorically rise to the occasion by being the best snack in the room.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc75-4407-6195-8003-b08dcfd27511'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:59:41.628661+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoin

### Fault Tolerance

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [ ]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [ ]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [ ]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))